# RPM별 응답 계산 — 교차 확인용 코랩 노트북

웹 도구(https://aebonlee.github.io/data09-04/)와 **같은 계산**을 파이썬으로 합니다. 결과를 웹 도구의 엑셀과 대조하는 용도입니다.

- 가진 주파수 f = 차수 × RPM / 60 (Hz) — 가정: 회전 차수
- 차수 응답 = |FRF(f)| × 가진력 (원문 참고 1)
- overall = √(Σ 차수 응답²) (원문 참고 2, RSS)

저장소의 `samples/예시데이터_FRF.csv` 는 **가상 예시 데이터**입니다.

## 1. FRF CSV 올리기
첫 행 머리행, 첫 열 주파수(Hz). 응답점 열은 `이름` / `이름 Mag`+`이름 Phase` / `이름 Real`+`이름 Imag` 중 하나로 찾습니다.

In [ ]:
from google.colab import files
up = files.upload()
FRF_CSV = next(iter(up))
FRF_CSV

In [ ]:
import argparse
import csv
import math
import sys


def read_frf(path, point):
    with open(path, encoding="utf-8-sig", newline="") as fp:
        rows = list(csv.reader(fp))
    head = [h.strip() for h in rows[0]]

    def col(name):
        return head.index(name) if name in head else -1

    if col(point) >= 0:
        fmt, a, b = "mag", col(point), -1
    elif col(point + " Mag") >= 0:
        fmt, a, b = "magphase", col(point + " Mag"), col(point + " Phase")
    elif col(point + " Real") >= 0:
        fmt, a, b = "reim", col(point + " Real"), col(point + " Imag")
    else:
        sys.exit("응답점 열을 찾지 못했습니다: " + point)
    recs = {}
    for r in rows[1:]:
        try:
            f = float(r[0])
            va = float(r[a])
            vb = float(r[b]) if b >= 0 else 0.0
        except (ValueError, IndexError):
            continue
        if f in recs:
            continue  # 같은 주파수는 처음 값만 (웹 도구와 같음)
        recs[f] = math.hypot(va, vb) if fmt == "reim" else abs(va)
    freq = sorted(recs)
    return freq, [recs[f] for f in freq]


def interpolate(freq, vals, f, method="linear"):
    if not (freq[0] <= f <= freq[-1]):
        return None
    lo, hi = 0, len(freq) - 1
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if freq[mid] <= f:
            lo = mid
        else:
            hi = mid
    if freq[lo] == f:
        return vals[lo]
    if freq[hi] == f:
        return vals[hi]
    f0, f1, v0, v1 = freq[lo], freq[hi], vals[lo], vals[hi]
    if method == "nearest":
        return v0 if f - f0 <= f1 - f else v1
    t = (f - f0) / (f1 - f0)
    if method == "loglog" and f0 > 0 and v0 > 0 and v1 > 0:
        tl = math.log(f / f0) / math.log(f1 / f0)
        return math.exp(math.log(v0) + tl * (math.log(v1) - math.log(v0)))
    return v0 + t * (v1 - v0)


def rpm_list(start, end, step):
    n = int(math.floor((end - start) / step + 1e-9)) + 1
    out = [round(start + i * step, 9) for i in range(n)]
    if round(out[-1], 6) < round(end, 6):
        out.append(end)
    return out




## 2. 조건 입력 — 웹 도구에 넣은 값과 같게

In [ ]:
POINT = '예시_운전석바닥_진동'   # 응답점 이름
ORDERS = [1, 2, 4]               # 차수
FORCES = [120, 60, 25]           # 차수별 가진력
RPM_START, RPM_END, RPM_STEP = 800, 3000, 50
INTERP = 'linear'                # linear / nearest / loglog

## 3. 계산

In [ ]:
freq, mag = read_frf(FRF_CSV, POINT)
rows = []
for rpm in rpm_list(RPM_START, RPM_END, RPM_STEP):
    comps = []
    for k, F in zip(ORDERS, FORCES):
        h = interpolate(freq, mag, k * rpm / 60.0, INTERP)
        comps.append(None if h is None else h * F)
    s = [c for c in comps if c is not None]
    rows.append([rpm] + comps + [math.sqrt(sum(c * c for c in s)) if s else None])
for r in rows[:10]:
    print(r)

In [ ]:
with open('rpm_response_result.csv', 'w', encoding='utf-8-sig', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['RPM'] + ['%g차 응답' % k for k in ORDERS] + ['overall'])
    w.writerows(rows)
files.download('rpm_response_result.csv')